In [1]:
import pandas as pd
import numpy as np
import h5py
##import standard modules

import pynbody
from pynbody.analysis import profile
##import pynbody charachteristics



In [2]:
import sys
directory_path  = '/home/vadilloj/MAP2023/Vadillo-Justice-League-Code'
sys.path.append(directory_path)
##add my path to the sys.path to make it easier to add in and find documents.


from Runnable_Modules import Base as base, IonUtils as ions, TrackingUtils as tracking
from Runnable_Modules.IonUtils import HI_factor, OVI_factor
from Runnable_Modules.Base import simulations
##import custom functions

In [3]:
##simulations is the pd data frame (in this case pre existing so imported from base) that stores the simulation charachteristics
##halo_chars is the pd data frame that stores the charachteristics of all halos within the simulations

In [4]:
tracking.find_halo_keys(simulations)
filenames  = ['Sandra', 'Ruth', 'Sonia', 'Elena']
simulations.insert(0, "gal num",  pd.Series(['h148', 'h229', 'h242','h329'], index = filenames))
simulations = simulations.reindex(filenames)

In [11]:
simulations

,gal num,filepath,Halo keys,mass,mstar,mgas,Rvir,nSats,HI_mass,OVI_mass,sat_mass_frac,sat_HI_frac,sat_OVI_frac
Sandra,h148,/home/vadilloj/MAP2023/Sims/h148.cosmo50PLK.30...,"[h148_2, h148_3, h148_4, h148_6, h148_7, h148_...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ruth,h229,/home/vadilloj/MAP2023/Sims/h229.cosmo50PLK.30...,"[h229_14, h229_18, h229_20, h229_22, h229_49]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sonia,h242,/home/vadilloj/MAP2023/Sims/h242.cosmo50PLK.30...,"[h242_8, h242_10, h242_21, h242_30, h242_38, h...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Elena,h329,/home/vadilloj/MAP2023/Sims/h329.cosmo50PLK.30...,"[h329_7, h329_29, h329_117]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,time,mstar,mgas,mass,Rvir,x,y,z,Hdist,sfr
key,,,,,,,,,,
h148_2,13.800797,2.268773e+09,8.252345e+09,9.575866e+10,95.146148,-3.128655,484.000000,-141.145879,504.170554,0.476290
h148_3,13.800797,1.502171e+09,2.975036e+09,4.697010e+10,76.051992,207.196879,197.000000,-55.850550,291.305391,0.098870
h148_4,13.800797,4.317847e+08,9.706413e+08,3.248971e+10,67.257466,139.893318,54.000000,-86.350237,173.039024,0.025735
h148_6,13.800797,3.315968e+08,1.010160e+09,2.869601e+10,64.527622,299.101876,74.000000,115.614250,329.096623,0.037392
h148_10,13.800797,1.353274e+08,1.139460e+09,1.055766e+10,46.245041,21.186472,32.000000,-47.891312,61.371365,0.022669
h148_12,13.800797,1.043767e+08,7.503453e+06,9.071870e+09,43.957874,-52.348853,16.000000,114.143760,126.590680,0.000137
h148_27,13.800797,8.415728e+07,1.026774e+08,3.249352e+09,31.223519,175.981794,1.000000,173.993626,247.476006,0.000535
h148_34,13.800797,5.759679e+06,5.869542e+07,2.628660e+09,29.083911,-176.110608,235.000000,63.534840,300.460683,0.000000
h148_38,13.800797,1.374093e+07,0.000000e+00,1.883522e+09,26.029436,-13.254242,29.980248,185.272730,188.150138,0.000000


In [7]:
# HChars.loc['h148_3'] = np.nan
# # HChars = HChars.drop('time', axis  = 1)

In [10]:
def get_chars(subsim,chars, key, trackedSubsim, IsSubsim = False):
    """
    gets the following charachteristics for the given subsim, and writes its charachteristics into the chars pd dictionary under key.
    'mass', 'mstar','mgas','Rvir',
    """
    
    chars.at[key, 'mass'] =np.sum(subsim['mass'])
    chars.at[key, 'mstar'] = np.sum(subsim.s['mass'])
    mass =  np.sum(subsim.g['mass'])
    OVI_mass =  np.sum(subsim.g['OVI_mass'])
    HI_mass =  np.sum(subsim.g['HI_mass'])
    chars.at[key, 'mgas'] = mass
    chars.at[key, 'OVI_mass'] = OVI_mass
    chars.at[key, 'HI_mass'] = HI_mass
    
    if IsSubsim:
        chars.at[key, 'host_mass_frac'] = (np.sum(trackedSubsim.g['mass'])/np.sum(h1.g['mass']))
        chars.at[key, 'host_HI_frac']  = (np.sum(trackedSubsim.g['HI_mass'])/np.sum(h1.g['HI_mass']))
        chars.at[key, 'host_OVI_frac']  = (np.sum(trackedSubsim.g['OVI_mass'])/np.sum(h1.g['OVI_mass']))
    else:
        chars.at[key, 'sat_mass_frac'] =  (np.sum(trackedSubsim.g['mass'])/chars['mass'][key])
        chars.at[key, 'sat_HI_frac']  = (np.sum(trackedSubsim.g['HI_mass'])/chars['HI_mass'][key])
        chars.at[key, 'sat_OVI_frac']  = (np.sum(trackedSubsim.g['OVI_mass'])/chars['OVI_mass'][key])

        chars.at[key, 'Rvir'] = np.max(subsim.g['r'])

In [12]:
def get_halo_chars(subsim, chars, key,tracked_subsim):
    warnings.filterwarnings("error")
    try:
        pos = pynbody.analysis.halo.center(subsim, return_cen = True).tolist() # recenters
    except RuntimeWarning:
        pos = [-1,-1,-1]
    warnings.resetwarnings()
    chars.at[key, 'x'] = round(pos[0], 2)
    chars.at[key, 'y'] = round(pos[1], 2)
    chars.at[key, 'z'] = round(pos[2], 2)
    chars.at[key, 'Hdist'] = round(np.sqrt(pos[0]**2+pos[1]**2+pos[2]**2), 2)
    

    with pynbody.analysis.center(subsim):
        chars.at[key, 'Rvir'] = np.max(subsim['r'])
    get_chars(subsim,chars, key, trackedSubsim, IsSubsim = True)
        
    chars.at[key, 'M_inhalo'] =  chars['mgas'][key]/(np.sum(trackedSubsim.g['mass']))
    chars.at[key, 'HI_inhalo']  = chars['HI_mass'][key]/(np.sum(trackedSubsim.g['HI_mass']))
    chars.at[key, 'OVI_inhalo']  = chars['OVI_mass'][key]/(np.sum(trackedSubsim.g['OVI_mass']))


In [ ]:
HChars_path = directory_path + '/' +"All_halo_charachteristics.csv"
sims_path = directory_path + '/' +"All_sim_charachteristics.csv"

In [ ]:
import warnings

for filename in ["Sandra"]:
    simulations.at[filename, 'nSats'] = len(simulations['Halo keys'][filename])
    s, h1, h = base.load_in_sim(filename, return_h = True)
    print("loaded in halo")
    ions.calculate_gas_mass(s)
    print("loaded in sims")
    subsims = tracking.find_halo_particles(h1, simulations, filename)
    get_chars(h1, simulations, filename, subsims['halos'])
    print(filename + " big chars done" )
    for key in simulations['Halo keys'][filename]:
        if key in list(HChars.index):
            halonum = (key.split('_')[1])
            halokey ="h"+halonum #string denoting the halo of origin
            trackedSubsim = subsims[halokey]
            
            get_halo_chars(h[int(halonum)], HChars, key,trackedSubsim)
    print(filename + " all done" )

loaded in halo


In [15]:

HChars.to_csv(HChars_path)
simulations.to_csv(sims_path)

In [20]:
HChars = pd.read_csv(HChars_path, index_col = 0)
simulations  = pd.read_csv(sims_path, index_col = 0)

In [28]:
tracking.find_halo_keys(simulations)

()

In [30]:
simulations

,gal num,filepath,Halo keys,mass,mstar,mgas,Rvir,nSats,HI_mass,OVI_mass,sat_mass_frac,sat_HI_frac,sat_OVI_frac
Sandra,h148,/home/vadilloj/MAP2023/Sims/h148.cosmo50PLK.30...,"[h148_2, h148_3, h148_4, h148_6, h148_7, h148_...",2.052077e+12,1.917510e+11,1.527100e+11,266.215150,299.0,1.414185e+10,2.609339e+06,0.008800,0.316318,0.093678
Ruth,h229,/home/vadilloj/MAP2023/Sims/h229.cosmo50PLK.30...,"[h229_14, h229_18, h229_20, h229_22, h229_49]",1.052426e+12,1.020237e+11,7.695528e+10,214.442995,5.0,1.862303e+10,1.403408e+06,0.000414,0.006439,0.006718
Sonia,h242,/home/vadilloj/MAP2023/Sims/h242.cosmo50PLK.30...,"[h242_8, h242_10, h242_21, h242_30, h242_38, h...",1.150238e+12,8.973881e+10,9.158841e+10,275.603382,7.0,2.162918e+10,1.155743e+06,0.007511,0.112475,0.085670
Elena,h329,/home/vadilloj/MAP2023/Sims/h329.cosmo50PLK.30...,"[h329_7, h329_29, h329_117]",7.104449e+11,8.991782e+10,2.673112e+10,188.010756,3.0,6.212192e+08,7.602241e+05,0.000727,0.114009,0.004595


In [17]:
sim_filepath = simulations['filepath']['Elena']

#load and set the units for the simulation
s = pynbody.load(sim_filepath)
s.physical_units() #  and ensure the units are correct
h = s.halos(halo_numbers='v1')

In [23]:
h[1].properties['pos']

SimArray([[ 1929.34069776, -4346.09018251,  1386.55435289],
          [ 1924.18936633, -4344.56504866,  1387.47496525],
          [ 1898.94344651, -4352.04990192,  1462.40172902],
          ...,
          [ 1877.10002058, -4442.10134431,  1476.1066995 ],
          [ 1876.37303018, -4441.91247209,  1476.94312031],
          [ 1877.12479376, -4441.75824507,  1476.53203452]],
         shape=(32638998, 3), 'kpc')

In [24]:
ions.calculate_gas_mass(s)

In [24]:
h[7].mean_by_mass('pos')

SimArray([ 53.90405548, -34.2847789 ,  15.94553042], 'kpc')

In [18]:
pynbody.analysis.halo.center(h[7], return_cen = True)

SimArray([ 53.52581653, -33.40925164,  15.57995842], 'kpc')

In [97]:
h[7].properties['Mvir']/1e9

np.float64(3.47607)

In [101]:
with pynbody.analysis.center(h[7]):
    print(np.max(h[7]['r']))

32.7381727217724


In [110]:
(np.max(h[7]['x']) - np.min(h[7]['x']))/2

SimArray(32.22719087, 'kpc')

In [12]:
pynbody.analysis.halo.center(h[7], return_cen = True).tolist()

[53.525816532036295, -33.409251635239954, 15.57995841779238]

In [88]:
sHaloChars.to_csv(data_name)

<Transformation sideon>

In [44]:
pynbody.analysis.angmom.sideon(s)

<Transformation sideon>

In [56]:
h[1].physical_units() 

In [122]:
s, h1, h = base.load_in_sim("Elena", return_h = True)

In [125]:
subsims = tracking.find_halo_particles(h1, simulations, 'Elena', groupSmalls = False  )##returns a dictionary of subsims for all halos in the simulations, 

In [127]:
len(subsims['halos'])

19821

In [128]:
len(subsims['halos'].g)

19821